##Cell 0 — Install (minimal, conflict-free)



In [ ]:
# Minimal stack for SAM fine-tuning via Hugging Face
!pip -q install "numpy>=2.0,<2.3" \
                "transformers>=4.44,<5" \
                nibabel \
                SimpleITK \
                datasets \
                opencv-python-headless==4.12.0.88 \
                scipy \
                git+https://github.com/facebookresearch/segment-anything.git


##Cell 1 — Mount Drive & base config

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, glob, random
import numpy as np
import torch

# Project paths
DRIVE_ROOT   = "/content/drive/MyDrive/ARNavigation"           # root containing case_01 ... case_15

# ---- NEW: Centralised outputs ----
OUTPUT_ROOT  = "/content/drive/MyDrive/ARNavigationSAMOutputs" # single place for all outputs
MODEL_DIR    = os.path.join(OUTPUT_ROOT, "Models")             # checkpoints go here
PREDICTION_DIR = os.path.join(OUTPUT_ROOT, "LesionPredictions")# centralised .nrrd predictions
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(PREDICTION_DIR, exist_ok=True)

# Dataset config
LESION_NAME = "Lesion.nrrd"
MOD_PREF = ["T1-CE.nii.gz", "T1-CE.nrrd", "T1WI.nii.gz", "T1WI.nrrd"]

# Reproducibility & device
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"

# Case glob used later
CASE_GLOB = os.path.join(DRIVE_ROOT, "case_*")



##Cell 2 — Imports & utility helpers (losses, I/O, metrics, viewer)

In [ ]:
# Core imports
import os, glob, json, math
import numpy as np
import nibabel as nib
import SimpleITK as sitk
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from transformers import SamProcessor, SamModel
import torch.nn.functional as F
import matplotlib.pyplot as plt
import cv2
from scipy.ndimage import binary_erosion, distance_transform_edt

# ---------- Losses (lightweight) ----------
class DiceBCELoss(torch.nn.Module):
    def __init__(self, dice_weight=0.5, bce_weight=0.5, smooth=1e-6):
        super().__init__()
        self.bce = torch.nn.BCEWithLogitsLoss()
        self.dw = dice_weight
        self.bw = bce_weight
        self.smooth = smooth
    def forward(self, logits, target):
        bce = self.bce(logits, target)
        probs = torch.sigmoid(logits)
        num = 2.0 * (probs * target).sum(dim=(1,2,3)) + self.smooth
        den = probs.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3)) + self.smooth
        dice = 1.0 - (num / den).mean()
        return self.dw * dice + self.bw * bce

class DiceFocalLoss(torch.nn.Module):
    def __init__(self, dice_weight=0.5, focal_weight=0.5, gamma=2.0, alpha=0.25, smooth=1e-6):
        super().__init__()
        self.dw, self.fw = dice_weight, focal_weight
        self.gamma, self.alpha, self.smooth = gamma, alpha, smooth
    def forward(self, logits, target):
        probs = torch.sigmoid(logits).clamp(1e-6, 1-1e-6)
        # focal (binary, logits-free form)
        pt = probs*target + (1-probs)*(1-target)
        focal = - (self.alpha*target*(1-probs)**self.gamma*torch.log(probs) +
                   (1-self.alpha)*(1-target)*probs**self.gamma*torch.log(1-probs))
        focal = focal.mean()
        # dice on probs
        num = 2.0 * (probs * target).sum(dim=(1,2,3)) + self.smooth
        den = probs.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3)) + self.smooth
        dice = 1.0 - (num / den).mean()
        return self.dw * dice + self.fw * focal

# ---------- I/O helpers ----------
def read_img(path):
    if path.endswith(".nii") or path.endswith(".nii.gz"):
        x = nib.load(path)
        arr = np.asanyarray(x.dataobj)          # (X,Y,Z) or (H,W,S)
        spacing = x.header.get_zooms()[:3]
        return np.moveaxis(arr, -1, 0), spacing # (S,H,W)
    else:  # .nrrd via SimpleITK
        img = sitk.ReadImage(path)
        arr = sitk.GetArrayFromImage(img)       # (S,H,W)
        spacing = img.GetSpacing()[::-1]
        return arr, spacing

def write_nrrd_like(ref_path, pred_3d_uint8, out_path):
    ref = sitk.ReadImage(ref_path)
    out = sitk.GetImageFromArray(pred_3d_uint8.astype(np.uint8))
    out.CopyInformation(ref)
    sitk.WriteImage(out, out_path)

def find_modality(case_dir):
    im_dir = os.path.join(case_dir, "imaging_data")
    for name in MOD_PREF:
        p = os.path.join(im_dir, name)
        if os.path.exists(p): return p
    cand = glob.glob(os.path.join(im_dir, "*.nii*")) + glob.glob(os.path.join(im_dir, "*.nrrd"))
    return cand[0] if cand else None

def load_case(case_dir):
    img_path = find_modality(case_dir)
    mask_path = os.path.join(case_dir, LESION_NAME)
    if not img_path: raise FileNotFoundError(f"No imaging in {case_dir}/imaging_data")
    if not os.path.exists(mask_path): raise FileNotFoundError(f"Missing {LESION_NAME} in {case_dir}")
    img_3d, _ = read_img(img_path)
    msk_3d, _ = read_img(mask_path)
    # resample image to mask grid if needed
    if img_3d.shape != msk_3d.shape:
        sitk_img = sitk.ReadImage(img_path)
        sitk_msk = sitk.ReadImage(mask_path)
        res = sitk.Resample(sitk_img, sitk_msk, sitk.Transform(), sitk.sitkLinear, 0.0, sitk_img.GetPixelID())
        img_3d = sitk.GetArrayFromImage(res)
    return img_3d, msk_3d, img_path, mask_path

def to_rgb3(slice2d):
    lo, hi = np.percentile(slice2d, 1), np.percentile(slice2d, 99)
    s = (slice2d - lo) / (hi - lo + 1e-6)
    s = np.clip(s, 0, 1)
    s = (s * 255).astype(np.uint8)
    return np.stack([s, s, s], axis=-1)

def bbox_from_mask(m):
    ys, xs = np.where(m > 0)
    if len(xs) == 0: return None
    x0, x1 = xs.min(), xs.max()
    y0, y1 = ys.min(), ys.max()
    pad = 5
    H, W = m.shape
    return [max(0, x0-pad), max(0, y0-pad), min(W-1, x1+pad), min(H-1, y1+pad)]

# ---------- Metrics (2D Dice & HD95 on logits grid) ----------
def dice2d(p, g, eps=1e-6):
    inter = (p & g).sum()
    return float((2*inter + eps) / (p.sum() + g.sum() + eps))

def hd95_2d(p, g):
    # p,g are binary uint8 arrays HxW (0/1). Pixel spacing unknown → pixels.
    p = p.astype(bool); g = g.astype(bool)
    if not p.any() and not g.any(): return 0.0
    if not p.any() and g.any():     # distance from empty to g surface
        g_surf = g ^ binary_erosion(g)
        return float(np.percentile(distance_transform_edt(~g_surf)[g_surf], 95))
    if p.any() and not g.any():
        p_surf = p ^ binary_erosion(p)
        return float(np.percentile(distance_transform_edt(~p_surf)[p_surf], 95))
    p_surf = p ^ binary_erosion(p)
    g_surf = g ^ binary_erosion(g)
    d1 = distance_transform_edt(~g_surf)[p_surf]
    d2 = distance_transform_edt(~p_surf)[g_surf]
    d  = np.hstack([d1, d2]) if d1.size and d2.size else (d1 if d1.size else d2)
    return float(np.percentile(d, 95)) if d.size else 0.0

# ---------- Confusion helpers (aggregated over pixels) ----------
def confusion_counts(pred_bin: np.ndarray, gt_bin: np.ndarray):
    """
    pred_bin, gt_bin: uint8 arrays of shape (...), values {0,1}
    Returns dict with tp, fp, tn, fn (ints)
    """
    p = pred_bin.astype(bool).ravel()
    g = gt_bin.astype(bool).ravel()
    tp = int(np.logical_and(p, g).sum())
    tn = int(np.logical_and(~p, ~g).sum())
    fp = int(np.logical_and(p, ~g).sum())
    fn = int(np.logical_and(~p, g).sum())
    return {"tp": tp, "tn": tn, "fp": fp, "fn": fn}

# ---------- Overlay viewer (reads from central predictions) ----------
def _norm01(img2d):
    lo, hi = np.percentile(img2d, 1), np.percentile(img2d, 99)
    x = (img2d - lo) / (hi - lo + 1e-6)
    return np.clip(x, 0, 1)

def _ensure_hw(arr, H, W, mode="nearest"):
    if arr.shape == (H, W): return arr
    t = torch.from_numpy(arr).float().unsqueeze(0).unsqueeze(0)
    t = F.interpolate(t, size=(H, W), mode=mode)
    return t.squeeze().cpu().numpy()

def _pick_best_slice(msk3d):
    areas = msk3d.reshape(msk3d.shape[0], -1).sum(axis=1)
    return int(areas.argmax()) if areas.max() > 0 else int(msk3d.shape[0] // 2)

def show_slice(case_dir, z=None, alpha=0.35, save=False):
    base = os.path.basename(case_dir)
    pred_path = os.path.join(PREDICTION_DIR, f"{base}_LesionPredictions.nrrd")
    if not os.path.exists(pred_path):
        raise FileNotFoundError(f"Prediction not found: {pred_path}; run inference first.")

    img3d, msk3d, _, mskp = load_case(case_dir)
    pred3d, _ = read_img(pred_path)

    S, H, W = img3d.shape
    if isinstance(z, str) and z.lower() == "auto": z = _pick_best_slice(msk3d)
    if z is None: z = _pick_best_slice(msk3d)
    z = int(np.clip(z, 0, S-1))

    img = img3d[z]
    gt  = (msk3d[z]  > 0).astype(np.uint8)
    pr  = (pred3d[z] > 0).astype(np.uint8)
    gt  = _ensure_hw(gt, H, W, "nearest")
    pr  = _ensure_hw(pr, H, W, "nearest")

    base_im = _norm01(img)
    inter = (gt * pr).sum()
    d_gt_pr  = (2*inter) / (gt.sum() + pr.sum() + 1e-6)

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(base_im, cmap="gray"); axes[0].set_title(f"Image (z={z})"); axes[0].axis("off")
    axes[1].imshow(base_im, cmap="gray"); axes[1].imshow(gt, cmap="Reds",   alpha=alpha); axes[1].set_title("Ground Truth"); axes[1].axis("off")
    axes[2].imshow(base_im, cmap="gray"); axes[2].imshow(pr, cmap="Greens", alpha=alpha); axes[2].set_title("Prediction");   axes[2].axis("off")
    axes[3].imshow(base_im, cmap="gray"); axes[3].imshow(gt, cmap="Reds", alpha=alpha); axes[3].imshow(pr, cmap="Greens", alpha=alpha)
    axes[3].set_title(f"Both (Dice={d_gt_pr*100:.1f}%)"); axes[3].axis("off")
    plt.tight_layout()
    if save:
        out_png = os.path.join(case_dir, f"overlay_z{z:03d}.png")
        plt.savefig(out_png, dpi=150, bbox_inches="tight")
        print("Saved:", out_png)
    plt.show()




##Cell 3 — Build train/val slice lists

In [ ]:
# Discover cases (sorted, stable)
cases = sorted([d for d in glob.glob(CASE_GLOB) if os.path.isdir(d)])
print(f"Found {len(cases)} cases.")

def harvest_slices(case_list, keep_empty=True):
    """
    Build slice items for a list of case directories.
    keep_empty=True includes negative slices with a full-slice prompt.
    """
    items = []
    for c in case_list:
        img3d, msk3d, _, mskp = load_case(c)
        S,H,W = img3d.shape
        for z in range(S):
            m = (msk3d[z]>0).astype(np.uint8)
            if not keep_empty and m.max()==0:
                continue
            rgb = to_rgb3(img3d[z])
            bb = bbox_from_mask(m) if m.max()>0 else [0,0,W-1,H-1]
            items.append({"image": Image.fromarray(rgb), "mask": m, "bbox": bb, "case": c, "z": z, "ref_mask_path": mskp})
    return items



##Cell 4 — Dataset + DataLoaders (+ custom collate)

In [ ]:
class SAMSliceDataset(Dataset):
    def __init__(self, items, processor, augment=False):
        self.items = items
        self.processor = processor
        self.augment = augment

        # --- Augmentation knobs (each < 0.5) ---
        self.p_hflip = 0.4
        self.p_vflip = 0.3
        self.p_rotate_scale_translate = 0.4  # composite affine

        # ranges
        self.max_deg = 15            # ± degrees
        self.scale_min, self.scale_max = 0.90, 1.10
        self.trans_frac = 0.05       # as fraction of width/height (±5%)

# THIS METHOD IS INSIDE THE `SAMSliceDataset` CLASS IN CELL 4
    def augment_pair(self, img_rgb, mask_u8):
        """
        Public method to apply the same augmentations used in training (for visualization).
        Returns:
            (aug_img, aug_mask, aug_description_string)
        """
        H, W = mask_u8.shape
        augs_applied = [] # <-- CHANGED: Track augmentations

        # flips
        if np.random.rand() < self.p_hflip:
            img_rgb = np.flip(img_rgb, axis=1).copy()
            mask_u8 = np.flip(mask_u8, axis=1).copy()
            augs_applied.append("HFlip") # <-- CHANGED

        if np.random.rand() < self.p_vflip:
            img_rgb = np.flip(img_rgb, axis=0).copy()
            mask_u8 = np.flip(mask_u8, axis=0).copy()
            augs_applied.append("VFlip") # <-- CHANGED

        # rotate + scale + translate
        if np.random.rand() < self.p_rotate_scale_translate:
            ang = np.random.uniform(-self.max_deg, self.max_deg)
            sc  = np.random.uniform(self.scale_min, self.scale_max)
            tx  = np.random.uniform(-self.trans_frac, self.trans_frac) * W
            ty  = np.random.uniform(-self.trans_frac, self.trans_frac) * H

            M = cv2.getRotationMatrix2D((W/2.0, H/2.0), ang, sc)
            M[:,2] += [tx, ty]

            img_rgb = cv2.warpAffine(img_rgb, M, (W, H), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101)
            mask_u8 = cv2.warpAffine(mask_u8, M, (W, H), flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT, borderValue=0)

            #
            augs_applied.append(f"Affine(R={ang:.1f}°,S={sc:.2f})")

        # ns
        if not augs_applied:
            augs_applied.append("None")

        aug_str = " + ".join(augs_applied)

        return img_rgb, mask_u8, aug_str # <

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        it = self.items[i]
        img_rgb = np.array(it["image"])           # HxWx3 uint8
        mask    = it["mask"].astype(np.uint8)     # HxW (0/1)

        # ---- AUGMENT (train only) ----
        if self.augment:
            img_rgb, mask = self.augment_pair(img_rgb, mask)

        # recompute bbox after any warp
        bb = bbox_from_mask(mask) if mask.max() > 0 else [0,0,img_rgb.shape[1]-1,img_rgb.shape[0]-1]

        proc = self.processor(Image.fromarray(img_rgb), input_boxes=[[bb]], return_tensors="pt")
        proc = {k:v.squeeze(0) for k,v in proc.items()}  # remove batch dim added by processor
        proc["ground_truth_mask"] = torch.tensor(mask, dtype=torch.float32)
        return proc

processor = SamProcessor.from_pretrained("facebook/sam-vit-base")

# Custom collate: stack pixel_values / input_boxes; keep GT as list (variable sizes)
def sam_collate(batch):
    pixel_values = torch.stack([b["pixel_values"] for b in batch], dim=0)  # (B,3,1024,1024)
    input_boxes  = torch.stack([b["input_boxes"]  for b in batch], dim=0)  # (B,1,4)
    gt_masks = [torch.as_tensor(b["ground_truth_mask"], dtype=torch.float32) for b in batch]
    return {"pixel_values": pixel_values, "input_boxes": input_boxes, "ground_truth_mask": gt_masks}




##Augmentation Viewer

In [ ]:
##Augmentation Viewer

# Visualize the effect of augmentations on random slices
import matplotlib.pyplot as plt
import random

def preview_augmentations(case_dir=None, num_examples=3, variants_per_example=4, seed=SEED):
    rng = np.random.RandomState(seed)
    random.seed(seed)

    # pick a case
    if case_dir is None:
        case_dir = random.choice(cases)
    img3d, msk3d, _, _ = load_case(case_dir)

    # build a tiny dataset just to borrow its augmentor
    dummy_items = []
    # choose random positive slices when available, otherwise random slice
    pos = np.where(msk3d.reshape(msk3d.shape[0], -1).sum(axis=1) > 0)[0]
    for _ in range(num_examples):
        if len(pos)>0:
            z = int(random.choice(list(pos)))
        else:
            z = int(rng.randint(0, img3d.shape[0]))
        rgb = to_rgb3(img3d[z])
        m   = (msk3d[z]>0).astype(np.uint8)
        dummy_items.append({"image": Image.fromarray(rgb), "mask": m})


    ds = SAMSliceDataset(dummy_items, processor, augment=False)

    # show grid
    cols = variants_per_example + 1
    fig, axes = plt.subplots(num_examples, cols, figsize=(4*cols, 4*num_examples)) #<-- Increased fig size for longer titles
    if num_examples == 1: axes = np.expand_dims(axes, 0)

    for r, it in enumerate(dummy_items):
        base = np.array(it["image"]); base_m = it["mask"]
        axes[r,0].imshow(base, cmap=None); axes[r,0].imshow(base_m, cmap="Reds", alpha=0.5)
        axes[r,0].set_title("Original"); axes[r,0].axis("off")

        for c in range(variants_per_example):
            # <-- CHANGED: Unpack the new title string
            aug_img, aug_m, aug_title = ds.augment_pair(base.copy(), base_m.copy())

            axes[r,c+1].imshow(aug_img, cmap=None); axes[r,c+1].imshow(aug_m, cmap="Greens", alpha=0.5)

            # <-- CHANGED: Use the new title string
            axes[r,c+1].set_title(aug_title);
            axes[r,c+1].axis("on")

    plt.tight_layout()
    plt.show()

# Example: preview 3 examples with 4 augmented variants each
preview_augmentations(num_examples=3, variants_per_example=4, seed=SEED)


##Bounding Boxes



In [ ]:
## Bounding Box Prompt Viewer (Per-Case, Best Positive vs. Negative)

import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Helper function to find the first negative slice in a 3D mask
def find_first_negative_slice(msk3d):
    """Returns the index of the first slice with no positive pixels."""
    if msk3d is None:
        return None
    for z in range(msk3d.shape[0]):
        if msk3d[z].max() == 0:
            return z
    return None # No negative slices found

print(f"Generating Bounding Box previews for all {len(cases)} cases...")

# Loop through every case
for case_dir in cases:
    try:
        img3d, msk3d, _, _ = load_case(case_dir)
        S, H, W = img3d.shape
    except FileNotFoundError:
        print(f"Skipping {os.path.basename(case_dir)} (files not found)")
        continue

    # --- 1. Find Best Positive Slice ---
    # We re-use the _pick_best_slice function from Cell 2
    z_pos_idx = _pick_best_slice(msk3d)
    m_pos = (msk3d[z_pos_idx] > 0).astype(np.uint8)

    pos_img, pos_bb = None, None
    if m_pos.max() > 0: # Check if a lesion actually exists on this slice
        pos_img = to_rgb3(img3d[z_pos_idx])
        pos_bb = bbox_from_mask(m_pos) # This will be a tight box

    # --- 2. Find a Negative Slice from the same case ---
    z_neg_idx = find_first_negative_slice(msk3d)

    neg_img, neg_bb = None, None
    if z_neg_idx is not None:
        neg_img = to_rgb3(img3d[z_neg_idx])
        neg_bb = [0, 0, W-1, H-1] # Full image box

    # --- 3. Plot this case ---
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    fig.suptitle(f"Case: {os.path.basename(case_dir)}", fontsize=16)

    # --- Plot Positive ---
    if pos_img is not None:
        axes[0].imshow(pos_img)
        axes[0].imshow(m_pos, cmap="Reds", alpha=0.3)
        rect_pos = patches.Rectangle(
            (pos_bb[0], pos_bb[1]),      # (x,y)
            pos_bb[2] - pos_bb[0],      # width
            pos_bb[3] - pos_bb[1],      # height
            linewidth=2, edgecolor='lime', facecolor='none'
        )
        axes[0].add_patch(rect_pos)
        axes[0].set_title(f"Slice (z={z_pos_idx})")
        axes[0].axis('off')
    else:
        axes[0].set_title("No positive lesion found in this case")
        axes[0].axis('off')

    # --- Plot Negative ---
    if neg_img is not None:
        axes[1].imshow(neg_img)
        rect_neg = patches.Rectangle(
            (neg_bb[0], neg_bb[1]),      # (x,y)
            neg_bb[2] - neg_bb[0],      # width
            neg_bb[3] - neg_bb[1],      # height
            linewidth=2, edgecolor='red', facecolor='none'
        )
        axes[1].add_patch(rect_neg)
        axes[1].set_title(f"Negative Slice Example (z={z_neg_idx})")
        axes[1].axis('off')
    else:
        axes[1].set_title("No negative slices found in this case")
        axes[1].axis('off')

    plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout for suptitle
    plt.show()

print("Done generating previews.")

##Cell 5 — Train (per-epoch Dice & HD95 on validation)


In [ ]:
from tqdm.auto import tqdm
import time
import numpy as np
import json

# ====== Option D knobs ======
USE_FOCAL = False            # set True to use Dice+Focal
THRESH    = 0.65             # decision threshold for metrics (0.5 → 0.65 reduces FPs)
DICE_W, BCE_W = 0.7, 0.3     # Dice/BCE weights (ignored if USE_FOCAL=True)
DICE_W_FOCAL, FOCAL_W = 0.5, 0.5
GAMMA, ALPHA = 2.0, 0.25

BATCH_SIZE = 2
EPOCHS     = 2
KFOLDS     = 5

def make_model_and_loss():
    model = SamModel.from_pretrained("facebook/sam-vit-base")
    # freeze encoder & prompt encoder; train mask decoder
    for name, p in model.named_parameters():
        if name.startswith("vision_encoder") or name.startswith("prompt_encoder"):
            p.requires_grad_(False)
    loss_fn = DiceFocalLoss(DICE_W_FOCAL, FOCAL_W, gamma=GAMMA, alpha=ALPHA) if USE_FOCAL else DiceBCELoss(DICE_W, BCE_W)
    return model.to(device), loss_fn

def make_optimizer(model, lr=1e-5, wd=0.0):
    return torch.optim.Adam(model.mask_decoder.parameters(), lr=lr, weight_decay=wd)

def _logits_from_output(outputs):
    logits = outputs.pred_masks
    if logits.ndim == 5:          # (B,1,1,Hl,Wl)
        logits = logits[:, 0, 0].unsqueeze(1)
    elif logits.ndim == 4:        # (B,1,Hl,Wl)
        pass
    else:
        raise RuntimeError(f"Unexpected pred_masks shape: {tuple(logits.shape)}")
    return logits

@torch.no_grad()
def evaluate_val_epoch(model, loader, device, thr=0.5, epoch=1, max_epochs=1):
    model.eval()
    dices, hd95s = [], []
    cm = {"tp":0, "tn":0, "fp":0, "fn":0}
    pbar = tqdm(loader, total=len(loader), desc=f"Eval   [{epoch}/{max_epochs}]", leave=False)
    for batch in pbar:
        pv = batch["pixel_values"].to(device)
        ib = batch["input_boxes"].to(device)
        gt_list = batch["ground_truth_mask"]
        out = model(pixel_values=pv, input_boxes=ib, multimask_output=False)
        logits = _logits_from_output(out)             # (B,1,Hl,Wl)
        Hl, Wl = logits.shape[-2:]
        probs = torch.sigmoid(logits).cpu().numpy()   # (B,1,Hl,Wl)
        for i, gt in enumerate(gt_list):
            gt = gt.numpy()
            # resize gt to (Hl,Wl)
            gt_t = torch.from_numpy(gt).float().unsqueeze(0).unsqueeze(0)
            gt_r = F.interpolate(gt_t, size=(Hl, Wl), mode="nearest").squeeze().numpy().astype(np.uint8)
            pr   = (probs[i,0] > thr).astype(np.uint8)
            dices.append(dice2d(pr, gt_r))
            hd = hd95_2d(pr, gt_r)
            hd95s.append(hd)
            c = confusion_counts(pr, gt_r)
            for k in cm: cm[k] += c[k]
        # live display of running means
        pbar.set_postfix(dice=f"{(np.mean(dices)*100 if dices else 0):.2f}%",
                         hd95px=f"{(np.mean(hd95s) if hd95s else 0):.2f}")
    dice_mean = float(np.mean(dices)) if dices else 0.0
    hd95_mean = float(np.mean(hd95s)) if hd95s else 0.0
    return dice_mean, hd95_mean, cm

def run_epoch(model, optimizer, loss_fn, loader, train=True, epoch=1, max_epochs=1):
    losses = []
    model.train() if train else model.eval()
    mode = "Train" if train else "Val"
    pbar = tqdm(loader, total=len(loader), desc=f"{mode}  [{epoch}/{max_epochs}]", leave=False)
    for batch in pbar:
        pixel_values = batch["pixel_values"].to(device)
        input_boxes  = batch["input_boxes"].to(device)
        gt_list      = batch["ground_truth_mask"]
        with torch.set_grad_enabled(train):
            out = model(pixel_values=pixel_values, input_boxes=input_boxes, multimask_output=False)
            logits = _logits_from_output(out)
            Hl, Wl = logits.shape[-2:]
            # resize and stack GT
            gtm = []
            for gt in gt_list:
                gt = gt.unsqueeze(0).unsqueeze(0).to(device)
                gt = F.interpolate(gt, size=(Hl, Wl), mode="nearest")
                gtm.append(gt)
            gtm = torch.cat(gtm, dim=0)
            loss = loss_fn(logits, gtm)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        loss_val = float(loss.item())
        losses.append(loss_val)
        pbar.set_postfix(loss=f"{loss_val:.4f}", avg=f"{(np.mean(losses)):.4f}")
    return float(np.mean(losses))

# ----- Create 5 folds by case (no leakage) -----
idx = np.arange(len(cases))
rng = np.random.RandomState(SEED)
perm = rng.permutation(idx)
folds = np.array_split(perm, KFOLDS)

all_fold_histories = []
best_overall = {"fold": None, "val_loss": float("inf"), "ckpt_path": None, "epoch": None}

t0_all = time.time()
for fi, val_idx in enumerate(folds, start=1):
    train_idx = np.setdiff1d(perm, val_idx)
    train_cases = [cases[i] for i in train_idx]
    val_cases   = [cases[i] for i in val_idx]
    print(f"\n=== Fold {fi}/{KFOLDS} ===")
    print(" Train cases:", [os.path.basename(c) for c in train_cases])
    print(" Val   cases:", [os.path.basename(c) for c in val_cases])

    # Build items
    train_items = harvest_slices(train_cases, keep_empty=True)
    val_items   = harvest_slices(val_cases,   keep_empty=True)

    # DataLoaders
    ds_train = SAMSliceDataset(train_items, processor, augment=True)
    ds_val   = SAMSliceDataset(val_items,   processor, augment=False)
    dl_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True,  drop_last=False, num_workers=0, collate_fn=sam_collate)
    dl_val   = DataLoader(ds_val,   batch_size=BATCH_SIZE, shuffle=False, drop_last=False, num_workers=0, collate_fn=sam_collate)

    # Model, loss, optim
    model, loss_fn = make_model_and_loss()
    optimizer = make_optimizer(model)

    # Training loop
    history = {"fold": fi, "train_loss":[], "val_loss":[], "val_dice":[], "val_hd95px":[], "val_cm":[]}
    best_val_loss = float("inf")
    ckpt_path = os.path.join(MODEL_DIR, f"tumour_sam_maskdec_fold{fi}.pth")

    for ep in range(1, EPOCHS+1):
        tr = run_epoch(model, optimizer, loss_fn, dl_train, train=True,  epoch=ep, max_epochs=EPOCHS)
        va = run_epoch(model, optimizer, loss_fn, dl_val,   train=False, epoch=ep, max_epochs=EPOCHS)
        dice_ep, hd95_ep, cm_ep = evaluate_val_epoch(model, dl_val, device, thr=THRESH, epoch=ep, max_epochs=EPOCHS)

        history["train_loss"].append(tr)
        history["val_loss"].append(va)
        history["val_dice"].append(dice_ep)
        history["val_hd95px"].append(hd95_ep)
        history["val_cm"].append(cm_ep)

        # log deltas vs previous epoch
        def dstr(curr, prev, fmt="{:+.4f}"):
            return "" if prev is None else fmt.format(curr - prev)
        prev_tr   = history["train_loss"][-2] if len(history["train_loss"])>1 else None
        prev_va   = history["val_loss"][-2]   if len(history["val_loss"])>1   else None
        prev_dice = history["val_dice"][-2]   if len(history["val_dice"])>1   else None
        prev_hd95 = history["val_hd95px"][-2] if len(history["val_hd95px"])>1 else None

        print(f"Fold {fi} | Epoch {ep:02d} | "
              f"train {tr:.4f}{' ' + dstr(tr, prev_tr) if prev_tr is not None else ''} | "
              f"val {va:.4f}{' ' + dstr(va, prev_va) if prev_va is not None else ''} | "
              f"Val Dice {dice_ep*100:.2f}%{' ' + dstr(dice_ep*100, (prev_dice*100 if prev_dice is not None else None), fmt='{:+.2f}%') if prev_dice is not None else ''} | "
              f"Val HD95 {hd95_ep:.2f} px{' ' + dstr(hd95_ep, prev_hd95, fmt='{:+.2f}') if prev_hd95 is not None else ''}")

        if va < best_val_loss:
            best_val_loss = va
            torch.save(model.state_dict(), ckpt_path)
            print(f"  ↳ Saved new best checkpoint to: {ckpt_path} (val_loss={best_val_loss:.4f})")

    # persist history for this fold
    hist_path = os.path.join(MODEL_DIR, f"history_fold{fi}.json")
    with open(hist_path, "w") as f:
        json.dump(history, f)
    print(f"Saved history: {hist_path}")

    all_fold_histories.append(history)

    # track overall best (by lowest val_loss across folds)
    if best_val_loss < best_overall["val_loss"]:
        best_overall = {"fold": fi, "val_loss": best_val_loss, "ckpt_path": ckpt_path, "epoch": int(np.argmin(history['val_loss'])+1)}

# summary
summary = {"best_overall": best_overall, "kfold": KFOLDS, "epochs": EPOCHS, "threshold": THRESH}
sum_path = os.path.join(MODEL_DIR, "kfold_summary.json")
with open(sum_path, "w") as f:
    json.dump(summary, f, indent=2)
print("\n=== K-Fold summary ===")
print(json.dumps(summary, indent=2))
print(f"Finished {KFOLDS}-fold CV in {(time.time()-t0_all)/60:.1f} min")




##Inference (baseline bbox prompting)

In [ ]:
# Reload best checkpoint (selected by lowest val loss across folds) and do simple bbox inference
# Uses GT bbox if available; else full-slice box

# Load summary
summary_path = os.path.join(MODEL_DIR, "kfold_summary.json")
with open(summary_path, "r") as f:
    ksum = json.load(f)

best_ckpt = ksum["best_overall"]["ckpt_path"]
best_fold = ksum["best_overall"]["fold"]
print(f"Using best fold {best_fold} checkpoint: {best_ckpt}")

# Build model
model = SamModel.from_pretrained("facebook/sam-vit-base")
for name, p in model.named_parameters():
    if name.startswith("vision_encoder") or name.startswith("prompt_encoder"):
        p.requires_grad_(False)
model.load_state_dict(torch.load(best_ckpt, map_location=device))
model.to(device).eval()

def _logits_from_output_single(outputs):
    logits = outputs.pred_masks
    if logits.ndim == 5:
        logits = logits[:, 0, 0]   # (1,Hl,Wl)
    elif logits.ndim == 4:
        logits = logits[:, 0]
    else:
        raise RuntimeError(f"Unexpected pred_masks shape: {tuple(logits.shape)}")
    return logits.unsqueeze(1)      # (1,1,Hl,Wl)

def infer_case(case_dir, thresh=0.65):
    base = os.path.basename(case_dir)  # e.g., case_01
    img3d, msk3d, _, mskp = load_case(case_dir)
    S,H,W = img3d.shape
    out = np.zeros((S,H,W), dtype=np.uint8)
    for z in range(S):
        rgb = to_rgb3(img3d[z])
        m = (msk3d[z] > 0).astype(np.uint8)
        bb = bbox_from_mask(m) if m.max()>0 else [0,0,W-1,H-1]
        inp = processor(Image.fromarray(rgb), input_boxes=[[bb]], return_tensors="pt")
        inp = {k:v.to(device) for k,v in inp.items()}
        with torch.no_grad():
            o = model(**inp, multimask_output=False)
            logits = _logits_from_output_single(o)              # (1,1,Hl,Wl)
            logits_up = F.interpolate(logits, size=(H, W), mode="bilinear", align_corners=False)
            prob = torch.sigmoid(logits_up)[0,0].cpu().numpy()
        out[z] = (prob > thresh).astype(np.uint8)

    out_path = os.path.join(PREDICTION_DIR, f"{base}_LesionPredictions.nrrd")
    write_nrrd_like(mskp, out, out_path)
    return out_path

# First, check how many cases we have
print(f"Total cases available: {len(cases)}")

# Process ALL cases
pred_paths = []
for i, case_dir in enumerate(cases):
    print(f"Processing case {i+1}/{len(cases)}: {case_dir}")
    pred_path = infer_case(case_dir, thresh=THRESH)
    pred_paths.append(pred_path)
    print(f"Completed: {pred_path}")

print(f"\nSuccessfully processed {len(pred_paths)} cases:")
for i, path in enumerate(pred_paths):
    print(f"Case {i+1}: {path}")



##Viewer

In [ ]:
# Example: largest tumour slice for a case
show_slice(cases[13], z="auto", alpha=0.35)

# Or sweep slices
case = cases[13]
img3d, _, _, _ = load_case(case)
for z in range(0, img3d.shape[0], 3):
    show_slice(case, z=z, alpha=0.35, save=False)




##Metrics and graphs


In [ ]:
##Metrics and graphs

import pandas as pd
from scipy.ndimage import binary_erosion, distance_transform_edt
import matplotlib.pyplot as plt # Make sure matplotlib is imported here

def dice3d(P, G, eps=1e-6):
    inter = (P & G).sum()
    return float((2*inter + eps) / (P.sum() + G.sum() + eps))

def _surface(mask):
    m = mask.astype(bool)
    if not m.any(): return m
    return m ^ binary_erosion(m)

def hd95_3d_mm(P, G, spacing):
    P = P.astype(bool); G = G.astype(bool)
    if not P.any() and not G.any(): return 0.0
    Ps = _surface(P); Gs = _surface(G)
    d_to_G = distance_transform_edt(~Gs, sampling=spacing)
    d_to_P = distance_transform_edt(~Ps, sampling=spacing)
    d1 = d_to_G[Ps]; d2 = d_to_P[Gs]
    if d1.size == 0 and d2.size == 0: return 0.0
    if d1.size == 0: return float(np.percentile(d2, 95))
    if d2.size == 0: return float(np.percentile(d1, 95))
    d = np.hstack([d1, d2]); return float(np.percentile(d, 95))

def assd_3d_mm(P, G, spacing):
    P = P.astype(bool); G = G.astype(bool)
    if not P.any() and not G.any(): return 0.0
    Ps = _surface(P); Gs = _surface(G)
    d_to_G = distance_transform_edt(~Gs, sampling=spacing)
    d_to_P = distance_transform_edt(~Ps, sampling=spacing)
    d1 = d_to_G[Ps]; d2 = d_to_P[Gs]
    if d1.size == 0 and d2.size == 0: return 0.0
    if d1.size == 0: return float(d2.mean())
    if d2.size == 0: return float(d1.mean())
    return float(0.5 * (d1.mean() + d2.mean()))

def volumetric_similarity(P, G, eps=1e-6):
    P = P.astype(bool); G = G.astype(bool)
    vp, vg = float(P.sum()), float(G.sum())
    return float(1.0 - abs(vp - vg) / (vp + vg + eps))

def cohen_kappa(P, G):
    p = P.astype(bool).ravel()
    g = G.astype(bool).ravel()
    N = p.size
    tp = np.logical_and(p, g).sum()
    tn = np.logical_and(~p, ~g).sum()
    fp = np.logical_and(p, ~g).sum()
    fn = np.logical_and(~p, g).sum()
    po = (tp + tn) / N
    pred_pos = (tp + fp) / N; pred_neg = (tn + fn) / N
    true_pos = (tp + fn) / N; true_neg = (tn + fp) / N
    pe = pred_pos * true_pos + pred_neg * true_neg
    if pe == 1.0: return 1.0
    return float((po - pe) / (1.0 - pe + 1e-12))

def eval_case(case_dir):
    base = os.path.basename(case_dir)
    gt_path   = os.path.join(case_dir, LESION_NAME)
    pred_path = os.path.join(PREDICTION_DIR, f"{base}_LesionPredictions.nrrd")
    if not os.path.exists(pred_path):
        return None
    gt, sp_gt   = read_img(gt_path)      # (S,H,W), spacing aligned (S,H,W) mm
    pr, _       = read_img(pred_path)

    if pr.shape != gt.shape:
        pr_t = torch.from_numpy(pr.astype(np.float32)).unsqueeze(0).unsqueeze(0)
        pr_r = F.interpolate(pr_t, size=gt.shape, mode="nearest").squeeze().numpy()
        pr = (pr_r > 0.5).astype(np.uint8)

    P = (pr>0).astype(np.uint8)
    G = (gt>0).astype(np.uint8)
    sp = tuple([float(s) for s in sp_gt]) if hasattr(sp_gt, "__len__") else (1.0,1.0,1.0)

    d3   = dice3d(P, G)
    hd95 = hd95_3d_mm(P, G, sp)
    assd = assd_3d_mm(P, G, sp)
    vs   = volumetric_similarity(P, G)
    kap  = cohen_kappa(P, G)
    vox_pred = int(P.sum()); vox_gt = int(G.sum())

    return {"case": base, "dice3d": d3, "hd95_mm": hd95, "assd_mm": assd,
            "volumetric_similarity": vs, "kappa": kap,
            "vox_pred": vox_pred, "vox_gt": vox_gt, "path": case_dir}

# run on all cases
case_list = cases
records = [r for r in (eval_case(c) for c in case_list) if r is not None]
df = pd.DataFrame(records).sort_values("dice3d", ascending=False).reset_index(drop=True)

# Display the full per-case results dataframe
display(df)

# =======================================================
# Generate Metric Summary Table
# =======================================================
print("\n" + "="*30)
print("     Metric Summary Table")
print("="*30)

# Define the metrics to summarize
metric_cols = [
    'dice3d',
    'hd95_mm',
    'assd_mm',
    'volumetric_similarity',
    'kappa',
    'vox_pred',
    'vox_gt'
]

# Calculate raw statistics
summary_table = df[metric_cols].agg(['mean', 'median', 'std', 'min', 'max']).T

# Rename columns for display
summary_table = summary_table.rename(columns={
    'std': 'Std Dev'
})

# Create the final formatted table
final_display_table = pd.DataFrame(index=summary_table.index)

# Apply specific formatting for each metric
final_display_table['Mean'] = [
    f"{summary_table.loc['dice3d', 'mean']*100:.2f}%",  # Dice
    f"{summary_table.loc['hd95_mm', 'mean']:.2f}",
    f"{summary_table.loc['assd_mm', 'mean']:.2f}",
    f"{summary_table.loc['volumetric_similarity', 'mean']:.3f}",
    f"{summary_table.loc['kappa', 'mean']:.3f}",
    f"{summary_table.loc['vox_pred', 'mean']:,.0f}",
    f"{summary_table.loc['vox_gt', 'mean']:,.0f}"
]

final_display_table['Median'] = [
    f"{summary_table.loc['dice3d', 'median']*100:.2f}%", # Dice
    f"{summary_table.loc['hd95_mm', 'median']:.2f}",
    f"{summary_table.loc['assd_mm', 'median']:.2f}",
    f"{summary_table.loc['volumetric_similarity', 'median']:.3f}",
    f"{summary_table.loc['kappa', 'median']:.3f}",
    f"{summary_table.loc['vox_pred', 'median']:,.0f}",
    f"{summary_table.loc['vox_gt', 'median']:,.0f}"
]

final_display_table['Std Dev'] = [
    f"{summary_table.loc['dice3d', 'Std Dev']:.3f}", # Std Dev is not a percentage
    f"{summary_table.loc['hd95_mm', 'Std Dev']:.2f}",
    f"{summary_table.loc['assd_mm', 'Std Dev']:.2f}",
    f"{summary_table.loc['volumetric_similarity', 'Std Dev']:.3f}",
    f"{summary_table.loc['kappa', 'Std Dev']:.3f}",
    f"{summary_table.loc['vox_pred', 'Std Dev']:,.1f}",
    f"{summary_table.loc['vox_gt', 'Std Dev']:,.1f}"
]

# Combine min and max into the requested 'range' column
final_display_table['Range (Min – Max)'] = [
    f"{summary_table.loc['dice3d', 'min']*100:.2f}% – {summary_table.loc['dice3d', 'max']*100:.2f}%",
    f"{summary_table.loc['hd95_mm', 'min']:.2f} – {summary_table.loc['hd95_mm', 'max']:.2f}",
    f"{summary_table.loc['assd_mm', 'min']:.2f} – {summary_table.loc['assd_mm', 'max']:.2f}",
    f"{summary_table.loc['volumetric_similarity', 'min']:.3f} – {summary_table.loc['volumetric_similarity', 'max']:.3f}",
    f"{summary_table.loc['kappa', 'min']:.3f} – {summary_table.loc['kappa', 'max']:.3f}",
    f"{summary_table.loc['vox_pred', 'min']:,.0f} – {summary_table.loc['vox_pred', 'max']:,.0f}",
    f"{summary_table.loc['vox_gt', 'min']:,.0f} – {summary_table.loc['vox_gt', 'max']:,.0f}"
]

# Rename index for better display
final_display_table.index.name = "Metric"
final_display_table = final_display_table.rename(index={
    'dice3d': 'Dice3D',
    'hd95_mm': 'HD95 (mm)',
    'assd_mm': 'ASSD (mm)',
    'volumetric_similarity': 'Volumetric Similarity',
    'kappa': 'Cohen\'s Kappa',
    'vox_pred': 'Predicted Voxels',
    'vox_gt': 'Ground Truth Voxels'
})

# Display the formatted table
display(final_display_table)
# =======================================================
# End of summary table code
# =======================================================


# =======================================================
# UPDATED: Generate Per-Case Plots (Separated)
# =======================================================

# --- Plot 1: Per-case Dice3D ---
plt.figure(figsize=(12, 5)) # Wider figure to spread bars
plt.bar(range(len(df)), df["dice3d"]*100, color='C0')
plt.xticks(range(len(df)), df["case"], rotation=45, ha='right')
plt.ylabel("Dice3D (%)")
plt.title("Per-case Dice3D")
plt.ylim(0, 100)
plt.grid(True, axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

# --- Plot 2: Per-case HD95 (mm) ---
plt.figure(figsize=(12, 5)) # Wider figure
plt.bar(range(len(df)), df["hd95_mm"], color='C1')
plt.xticks(range(len(df)), df["case"], rotation=45, ha='right')
plt.ylabel("HD95 (mm)")
plt.title("Per-case HD95 (mm)")
plt.grid(True, axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

# --- Plot 3: Per-case ASSD (mm) ---
plt.figure(figsize=(12, 5)) # Wider figure
plt.bar(range(len(df)), df["assd_mm"], color='C2')
plt.xticks(range(len(df)), df["case"], rotation=45, ha='right')
plt.ylabel("ASSD (mm)")
plt.title("Per-case ASSD (mm)")
plt.grid(True, axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

# =======================================================
# End of plots
# =======================================================

# Best / worst by Dice
best_row  = df.iloc[0]
worst_row = df.iloc[-1]
print("\nBest case:", best_row["case"],
      f"Dice3D={best_row['dice3d']*100:.2f}%, HD95={best_row['hd95_mm']:.2f} mm, ASSD={best_row['assd_mm']:.2f} mm, VS={best_row['volumetric_similarity']:.3f}, κ={best_row['kappa']:.3f}")
print("Worst case:", worst_row["case"],
      f"Dice3D={worst_row['dice3d']*100:.2f}%, HD95={worst_row['hd95_mm']:.2f} mm, ASSD={worst_row['assd_mm']:.2f} mm, VS={worst_row['volumetric_similarity']:.3f}, κ={worst_row['kappa']:.3f}")

In [ ]:
!pip -q install scikit-image trimesh networkx

import os, numpy as np, SimpleITK as sitk
from skimage import measure
import trimesh
from scipy.ndimage import label as cc_label

OUT_ROOT = OUTPUT_ROOT
OUT_DIR  = os.path.join(OUT_ROOT, "LesionOBJs")
os.makedirs(OUT_DIR, exist_ok=True)

# ---------- Controls ----------
SCALE_FACTOR = 1.0
EXPORT_COMPONENTS = "all"     # "largest" or "all"
MIN_COMPONENT_VOXELS = 500
SHIFT_PRED_ALONG_X = 0.0

def _components(mask_bool):
    lab, n = cc_label(mask_bool)
    if n == 0:
        return []
    sizes = np.bincount(lab.ravel())
    sizes[0] = 0
    if EXPORT_COMPONENTS == "largest":
        k = int(np.argmax(sizes))
        return [(k, lab == k)]
    keep = []
    for k in range(1, len(sizes)):
        if sizes[k] >= MIN_COMPONENT_VOXELS:
            keep.append((k, lab == k))
    return keep

def _mesh_from_mask(mask_bool, sitk_img, scale=SCALE_FACTOR, translate=None):
    verts_zyx, faces, _, _ = measure.marching_cubes(mask_bool.astype(np.uint8), level=0.5, step_size=1)
    verts_xyz = np.stack([verts_zyx[:,2], verts_zyx[:,1], verts_zyx[:,0]], axis=1)

    spacing   = np.array(sitk_img.GetSpacing())                 # (sx,sy,sz) in mm
    origin    = np.array(sitk_img.GetOrigin())                  # (ox,oy,oz) in mm
    direction = np.array(sitk_img.GetDirection()).reshape(3,3)  # 3×3

    verts_mm   = (direction @ (verts_xyz * spacing).T).T + origin
    verts_out  = verts_mm * scale
    if translate is not None:
        verts_out = verts_out + np.asarray(translate)
    return trimesh.Trimesh(vertices=verts_out, faces=faces, process=False)

def _export_all_from_nrrd(nrrd_path, stem, shift_pred=False):
    img = sitk.ReadImage(nrrd_path)
    vol = sitk.GetArrayFromImage(img)  # (z,y,x)

    labels = np.unique(vol)
    labels = labels[labels > 0]
    if labels.size == 0 and vol.max() > 0:
        labels = np.array([1], dtype=vol.dtype)
    if labels.size == 0:
        print(f"[skip] Empty: {os.path.basename(nrrd_path)}")
        return 0

    exported = 0
    for lbl in labels:
        mask = (vol == lbl) if vol.max() > 1 else (vol > 0)
        comps = _components(mask.astype(bool))
        if not comps:
            continue
        for ci, (k, comp) in enumerate(comps, start=1):
            translate = (SHIFT_PRED_ALONG_X, 0, 0) if shift_pred else None
            mesh = _mesh_from_mask(comp, img, translate=translate)
            out_path = os.path.join(OUT_DIR, f"{stem}__L{int(lbl)}__C{ci:02d}.obj")
            mesh.export(out_path)
            print("  →", out_path, f"(V={len(mesh.vertices)} F={len(mesh.faces)})")
            exported += 1
    return exported

total = 0
for case_dir in cases:
    base = os.path.basename(case_dir)

    # Predicted (from central folder)
    pred_path = os.path.join(PREDICTION_DIR, f"{base}_LesionPredictions.nrrd")
    if os.path.exists(pred_path):
        total += _export_all_from_nrrd(pred_path, f"{base}__pred", shift_pred=(SHIFT_PRED_ALONG_X != 0.0))
    else:
        print(f"[note] No prediction for {base}")

    # Ground truth (from case folder)
    gt_path = os.path.join(case_dir, LESION_NAME)
    if os.path.exists(gt_path):
        total += _export_all_from_nrrd(gt_path, f"{base}__gt", shift_pred=False)

print(f"\nDone. Saved OBJ files in: {OUT_DIR}\nTotal OBJ exports: {total}")





##Generic Metrics

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import os

# Load all fold histories if not already in memory
histories = []
for fi in range(1, 6):
    hp = os.path.join(MODEL_DIR, f"history_fold{fi}.json")
    if os.path.exists(hp):
        with open(hp, "r") as f:
            histories.append(json.load(f))

if not histories:
    # fallback to in-memory from Cell 5 if present
    try:
        histories = all_fold_histories
    except NameError:
        histories = []

assert histories, "No histories found. Train (Cell 5) first."

# Find max epochs across all folds
max_epochs = 0
for h in histories:
    max_epochs = max(max_epochs, len(h["train_loss"]))
xs_max = np.arange(1, max_epochs + 1)

print(f"Loaded {len(histories)} history files. Max epochs: {max_epochs}")

# ---------- NEW: Plot 1 - Loss per Fold (Separate Subplots) ----------
fig1, axes1 = plt.subplots(2, 3, figsize=(18, 10))
axes1_flat = axes1.flatten()
print("Generating fold_loss_curves.png...")

for i, h in enumerate(histories):
    ax = axes1_flat[i]
    fold_epochs = len(h['train_loss'])
    xs_fold = np.arange(1, fold_epochs + 1)

    ax.plot(xs_fold, h['train_loss'], label="Train Loss", color='C0', marker='o', markersize=4)
    ax.plot(xs_fold, h['val_loss'], label="Val Loss", linestyle="--", color='C1', marker='x', markersize=4)
    ax.set_title(f"Fold {h['fold']} Loss")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xticks(xs_fold) # Ensure ticks for each epoch if few epochs

# Hide any unused subplots (if KFOLDS < 6)
for j in range(len(histories), 6):
    fig1.delaxes(axes1_flat[j])

fig1.tight_layout()
fig1.savefig("fold_loss_curves.png", dpi=100)
plt.close(fig1) # Close figure to save memory

# ---------- NEW: Plot 2 - Validation Dice per Fold (Separate Subplots) ----------
fig2, axes2 = plt.subplots(2, 3, figsize=(18, 10))
axes2_flat = axes2.flatten()
print("Generating fold_dice_curves.png...")

for i, h in enumerate(histories):
    ax = axes2_flat[i]
    fold_epochs = len(h['val_dice'])
    xs_fold = np.arange(1, fold_epochs + 1)

    ax.plot(xs_fold, np.array(h['val_dice']) * 100, label="Val Dice", color='C2', marker='o', markersize=4)
    ax.set_title(f"Fold {h['fold']} Validation Dice")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Dice (%)")
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 100)
    ax.set_xticks(xs_fold)

##Generic Metrics

import json
import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd # <-- ADDED IMPORT

# Load all fold histories if not already in memory
histories = []
for fi in range(1, 6):
    hp = os.path.join(MODEL_DIR, f"history_fold{fi}.json")
    if os.path.exists(hp):
        with open(hp, "r") as f:
            histories.append(json.load(f))

if not histories:
    # fallback to in-memory from Cell 5 if present
    try:
        histories = all_fold_histories
    except NameError:
        histories = []

assert histories, "No histories found. Train (Cell 5) first."

# ---------- Plot loss curves per fold and averaged ----------
max_epochs = max(len(h["train_loss"]) for h in histories)
xs_max = np.arange(1, max_epochs + 1)

print(f"Loaded {len(histories)} history files. Max epochs: {max_epochs}")

# ---------- NEW: Plot 1 - Loss per Fold (Separate Subplots) ----------
fig1, axes1 = plt.subplots(2, 3, figsize=(18, 10))
axes1_flat = axes1.flatten()
print("Generating fold_loss_curves.png...")

for i, h in enumerate(histories):
    ax = axes1_flat[i]
    fold_epochs = len(h['train_loss'])
    xs_fold = np.arange(1, fold_epochs + 1)

    ax.plot(xs_fold, h['train_loss'], label="Train Loss", color='C0', marker='o', markersize=4)
    ax.plot(xs_fold, h['val_loss'], label="Val Loss", linestyle="--", color='C1', marker='x', markersize=4)
    ax.set_title(f"Fold {h['fold']} Loss")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xticks(xs_fold) # Ensure ticks for each epoch if few epochs

# Hide any unused subplots (if KFOLDS < 6)
for j in range(len(histories), 6):
    fig1.delaxes(axes1_flat[j])

fig1.tight_layout()
fig1.savefig("fold_loss_curves.png", dpi=100)
plt.close(fig1) # Close figure to save memory

# ---------- NEW: Plot 2 - Validation Dice per Fold (Separate Subplots) ----------
fig2, axes2 = plt.subplots(2, 3, figsize=(18, 10))
axes2_flat = axes2.flatten()
print("Generating fold_dice_curves.png...")

for i, h in enumerate(histories):
    ax = axes2_flat[i]
    fold_epochs = len(h['val_dice'])
    xs_fold = np.arange(1, fold_epochs + 1)

    ax.plot(xs_fold, np.array(h['val_dice']) * 100, label="Val Dice", color='C2', marker='o', markersize=4)
    ax.set_title(f"Fold {h['fold']} Validation Dice")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Dice (%)")
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 100)
    ax.set_xticks(xs_fold)

# Hide any unused subplots
for j in range(len(histories), 6):
    fig2.delaxes(axes2_flat[j])

fig2.tight_layout()
fig2.savefig("fold_dice_curves.png", dpi=100)
plt.close(fig2)

# ---------- NEW: Plot 3 - Validation HD95 per Fold (Separate Subplots) ----------
fig3, axes3 = plt.subplots(2, 3, figsize=(18, 10))
axes3_flat = axes3.flatten()
print("Generating fold_hd95_curves.png...")

for i, h in enumerate(histories):
    ax = axes3_flat[i]
    fold_epochs = len(h['val_hd95px'])
    xs_fold = np.arange(1, fold_epochs + 1)

    ax.plot(xs_fold, h['val_hd95px'], label="Val HD95 (px)", color='C3', marker='o', markersize=4)
    ax.set_title(f"Fold {h['fold']} Validation HD95")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("HD95 (pixels)")
    ax.legend()
    ax.grid(True, alpha=0.3)
    # Autoscale Y-axis for HD95, but set x-ticks
    ax.set_xticks(xs_fold)

# Hide any unused subplots
for j in range(len(histories), 6):
    fig3.delaxes(axes3_flat[j])

fig3.tight_layout()
fig3.savefig("fold_hd9f_curves.png", dpi=100)
plt.close(fig3)

# ---------- Average curves (as before, but saving figs) ----------
def pad_to(hlist, L):
    """Pads a list of lists to length L by repeating the last element."""
    padded = []
    for hl in hlist:
        if not hl: # Handle empty history list
            padded.append([0.0] * L) # Or np.nan
        else:
            padded.append(hl + [hl[-1]] * (L - len(hl)))
    return np.array(padded, dtype=float)

# Average Loss
avg_train = pad_to([h["train_loss"] for h in histories], max_epochs).mean(axis=0)
avg_val = pad_to([h["val_loss"] for h in histories], max_epochs).mean(axis=0)

fig_avg_loss = plt.figure(figsize=(10, 5))
plt.plot(xs_max, avg_train, label="Avg Train Loss", marker='o')
plt.plot(xs_max, avg_val, label="Avg Val Loss", linestyle="--", marker='x')
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Average Loss Curves (across folds)")
plt.legend(); plt.grid(True, alpha=0.3); plt.xticks(xs_max)
fig_avg_loss.savefig("average_loss_curve.png", dpi=100)
plt.close(fig_avg_loss)
print("Generating average_loss_curve.png...")

# Average Dice
avg_dice = pad_to([h["val_dice"] for h in histories], max_epochs).mean(axis=0)
fig_avg_dice = plt.figure(figsize=(7, 5))
plt.plot(xs_max, avg_dice * 100, marker='o', color='C2')
plt.xlabel("Epoch"); plt.ylabel("Val Dice (%)"); plt.title("Average Val Dice (across folds)")
plt.grid(True, alpha=0.3); plt.ylim(0, 100); plt.xticks(xs_max)
fig_avg_dice.savefig("average_dice_curve.png", dpi=100)
plt.close(fig_avg_dice)
print("Generating average_dice_curve.png...")


# ---------- UPDATED: Confusion matrix (from best fold, last epoch) ----------
print("Generating confusion_matrix.png...")
with open(os.path.join(MODEL_DIR, "kfold_summary.json"), "r") as f:
    ksum = json.load(f)
best_fold = ksum["best_overall"]["fold"]
THRESH = ksum["threshold"] # <-- Fixed NameError here

best_hist = None
for h in histories:
    if int(h["fold"]) == int(best_fold):
        best_hist = h; break

assert best_hist is not None, "Best fold history not found."

# Get CM from the last epoch
cm = best_hist["val_cm"][-1]
tp, tn, fp, fn = cm["tp"], cm["tn"], cm["fp"], cm["fn"]

cm_mat = np.array([[tp, fp],
                   [fn, tn]], dtype=float)

# --- Plotting with new formatting ---
fig_cm, axes_cm = plt.subplots(1, 2, figsize=(13, 5)) # Slightly wider

# --- Plot 1: Raw counts ---
ax_raw = axes_cm[0]
im_raw = ax_raw.imshow(cm_mat, interpolation='nearest', cmap='Blues')
ax_raw.set_title(f"Confusion Matrix (Fold {best_fold}, last epoch)")
ax_raw.set_xticks([0, 1]); ax_raw.set_xticklabels(["Pred Pos", "Pred Neg"])
ax_raw.set_yticks([0, 1]); ax_raw.set_yticklabels(["True Pos", "True Neg"])

# Add text with dynamic color
thresh_raw = cm_mat.max() / 2.
for i in range(2):
    for j in range(2):
        ax_raw.text(j, i, f"{int(cm_mat[i,j]):,}", # Add comma formatting
                    ha="center", va="center",
                    color="white" if cm_mat[i, j] > thresh_raw else "black",
                    fontsize=12, fontweight="medium")

fig_cm.colorbar(im_raw, ax=ax_raw, fraction=0.045, pad=0.04)

# --- Plot 2: Normalized (by true class) ---
ax_norm = axes_cm[1]
cm_norm = cm_mat / (cm_mat.sum(axis=1, keepdims=True) + 1e-9)
im_norm = ax_norm.imshow(cm_norm, interpolation='nearest', cmap='Blues', vmin=0, vmax=1)
ax_norm.set_title("Confusion Matrix (Normalized by True Class)")
ax_norm.set_xticks([0, 1]); ax_norm.set_xticklabels(["Pred Pos", "Pred Neg"])
ax_norm.set_yticks([0, 1]); ax_norm.set_yticklabels(["True Pos", "True Neg"])

# Add text with dynamic color
thresh_norm = 0.5 # Normalized data is 0-1
for i in range(2):
    for j in range(2):
        ax_norm.text(j, i, f"{cm_norm[i,j]*100:.1f}%",
                     ha="center", va="center",
                     color="white" if cm_norm[i,j] > thresh_norm else "black",
                     fontsize=12, fontweight="medium")

fig_cm.colorbar(im_norm, ax=ax_norm, fraction=0.045, pad=0.04)

fig_cm.tight_layout()
fig_cm.savefig("confusion_matrix.png", dpi=100)
plt.close(fig_cm)


# =======================================================
# NEW: Derived Metrics Table
# =======================================================
# Calculate metrics
accuracy    = (tp + tn) / (tp + tn + fp + fn + 1e-9)
precision   = tp / (tp + fp + 1e-9)
recall      = tp / (tp + fn + 1e-9)  # Sensitivity / TPR
specificity = tn / (tn + fp + 1e-9) # Specificity / TNR
f1          = 2 * precision * recall / (precision + recall + 1e-9)

# Create dictionary for the table
metric_data = {
    "Metric": [
        "Pixel Accuracy",
        "Precision (PPV)",
        "Recall (Sensitivity/TPR)",
        "Specificity (TNR)",
        "F1 Score"
    ],
    "Value": [
        accuracy,
        precision,
        recall,
        specificity,
        f1
    ]
}

# Create and format the DataFrame
metrics_df = pd.DataFrame(metric_data)
metrics_df = metrics_df.set_index("Metric")
metrics_df['Value'] = metrics_df['Value'].map('{:.4f}'.format) # Apply formatting

# Get epoch number for the title
last_epoch_num = len(best_hist['val_cm'])

print("\n" + "="*50)
print(f"--- Derived Metrics (Fold {best_fold}, Last Epoch {last_epoch_num}) ---")
print("="*50)

# Display the formatted table
display(metrics_df)

# Print remaining info
print(f"\n(Threshold used for metrics: THRESH={THRESH})")
print(f"Raw Counts: TP={tp:,} | TN={tn:,} | FP={fp:,} | FN={fn:,}")

In [ ]:
# ---------- NEW: Table — 2D Val Dice per Fold + Averages ----------
import numpy as np
import pandas as pd

records = []
for h in histories:
    fold_id = int(h.get("fold", len(records)+1))
    dice_arr = np.array(h["val_dice"], dtype=float)  # assumed 0..1
    last_dice = dice_arr[-1] * 100.0
    best_idx = int(np.argmax(dice_arr))
    best_dice = dice_arr[best_idx] * 100.0
    records.append({
        "Fold": fold_id,
        "Val Dice (Last %)": last_dice,
        "Val Dice (Best %)": best_dice,
        "Epoch of Best": best_idx + 1
    })

# Sort by fold for neatness
dice_df = pd.DataFrame(records).sort_values("Fold").reset_index(drop=True)

# Compute averages across folds
avg_last = dice_df["Val Dice (Last %)"].mean()
avg_best = dice_df["Val Dice (Best %)"].mean()

avg_row = {
    "Fold": "Average",
    "Val Dice (Last %)": avg_last,
    "Val Dice (Best %)": avg_best,
    "Epoch of Best": ""
}

dice_df = pd.concat([dice_df, pd.DataFrame([avg_row])], ignore_index=True)

# Nicely formatted display
with pd.option_context('display.float_format', '{:,.2f}'.format):
    print("\n=== 2D Validation Dice per Fold (+ Averages) ===")
    print(dice_df.to_string(index=False))

# Save to CSV (and optional LaTeX table for your report)
out_csv = os.path.join(MODEL_DIR, "val_dice_per_fold.csv")
dice_df.to_csv(out_csv, index=False)
print(f"\nSaved: {out_csv}")

# Optional LaTeX export (uncomment if you want a .tex table too)
# out_tex = os.path.join(MODEL_DIR, "val_dice_per_fold.tex")
# with open(out_tex, "w") as f:
#     f.write(dice_df.to_latex(index=False, float_format="%.2f"))
# print(f"Saved: {out_tex}")


In [ ]:
# === Exact Training + Validation Loss per Fold (tables) ===
import os, json, numpy as np, pandas as pd

# 1) Load histories
histories = []
for fi in range(1, 6):
    hp = os.path.join(MODEL_DIR, f"history_fold{fi}.json")
    if os.path.exists(hp):
        with open(hp, "r") as f:
            h = json.load(f)
            h["fold"] = int(h.get("fold", fi))
            histories.append(h)

if not histories:
    try:
        histories = all_fold_histories
    except NameError:
        raise RuntimeError("No histories found. Train first or ensure MODEL_DIR is correct.")

# 2) Optional: load kfold summary for test losses
ksum_path = os.path.join(MODEL_DIR, "kfold_summary.json")
ksum = None
if os.path.exists(ksum_path):
    with open(ksum_path, "r") as f:
        ksum = json.load(f)

def find_test_loss_for_fold(fold: int):
    """Return a single test loss for this fold if available, else NaN."""
    h = next((hh for hh in histories if int(hh.get("fold", -1)) == int(fold)), None)
    if h:
        for k in ["test_loss", "test_loss_final"]:
            if k in h and isinstance(h[k], (int, float)):
                return float(h[k])
        if "test_loss" in h and isinstance(h["test_loss"], (list, tuple)) and len(h["test_loss"]) > 0:
            return float(h["test_loss"][-1])

    tfp = os.path.join(MODEL_DIR, f"test_metrics_fold{fold}.json")
    if os.path.exists(tfp):
        with open(tfp, "r") as f:
            tm = json.load(f)
        for k in ["test_loss", "loss", "avg_loss", "mean_loss"]:
            if k in tm and isinstance(tm[k], (int, float)):
                return float(tm[k])

    if ksum:
        # common nested shapes
        try:
            return float(ksum["folds"][str(fold)]["test"]["loss"])
        except Exception:
            pass
        try:
            return float(ksum["folds"][fold]["test"]["loss"])
        except Exception:
            pass
        try:
            per_fold = ksum.get("per_fold", [])
            for item in per_fold:
                if int(item.get("fold", -1)) == int(fold):
                    for k in ["test_loss", "loss", "avg_loss", "mean_loss"]:
                        if k in item:
                            return float(item[k])
                    if "test" in item and isinstance(item["test"], dict):
                        for k in ["loss", "test_loss"]:
                            if k in item["test"]:
                                return float(item["test"][k])
        except Exception:
            pass
        for k, v in ksum.items():
            if f"fold{fold}" in k and ("test" in k and "loss" in k) and isinstance(v, (int, float)):
                return float(v)

    return np.nan

# 3) Long table: Fold, Epoch, TrainLoss, ValLoss
rows_long = []
for h in histories:
    fold = int(h["fold"])
    tloss = [float(x) for x in h.get("train_loss", [])]
    vloss = [float(x) for x in h.get("val_loss", [])]
    max_len = max(len(tloss), len(vloss))
    for e in range(1, max_len + 1):
        t = tloss[e-1] if e <= len(tloss) else np.nan
        v = vloss[e-1] if e <= len(vloss) else np.nan
        rows_long.append({"Fold": fold, "Epoch": e, "TrainLoss": t, "ValLoss": v})

loss_long_df = pd.DataFrame(rows_long).sort_values(["Fold", "Epoch"]).reset_index(drop=True)

# 4) Wide table: Train_E*, Val_E*, plus TestLoss
max_epochs = 0
for h in histories:
    max_epochs = max(max_epochs, len(h.get("train_loss", [])), len(h.get("val_loss", [])))

wide_rows = []
for h in histories:
    fold = int(h["fold"])
    tl = [float(x) for x in h.get("train_loss", [])]
    vl = [float(x) for x in h.get("val_loss", [])]
    row = {"Fold": fold}
    for e in range(1, max_epochs + 1):
        row[f"Train_E{e}"] = tl[e-1] if e <= len(tl) else np.nan
        row[f"Val_E{e}"]   = vl[e-1] if e <= len(vl) else np.nan
    row["TestLoss"] = find_test_loss_for_fold(fold)
    wide_rows.append(row)

loss_wide_df = pd.DataFrame(wide_rows).sort_values("Fold").reset_index(drop=True)

# 5) Summary tied to BEST validation epoch: include train loss at that epoch
summ_rows = []
for h in histories:
    fold = int(h["fold"])
    vl = np.array(h.get("val_loss", []), dtype=float)
    tl = np.array(h.get("train_loss", []), dtype=float)
    if len(vl):
        best_e = int(np.nanargmin(vl) + 1)
        best_v = float(np.nanmin(vl))
        train_at_best = float(tl[best_e - 1]) if best_e - 1 < len(tl) else np.nan
    else:
        best_e, best_v, train_at_best = np.nan, np.nan, np.nan
    summ_rows.append({
        "Fold": fold,
        "BestValEpoch": best_e,
        "BestValLoss": best_v,
        "TrainLoss_at_BestVal": train_at_best,
        "TestLoss": find_test_loss_for_fold(fold)
    })

val_best_df = pd.DataFrame(summ_rows).sort_values("Fold").reset_index(drop=True)

# 6) Save CSVs
out_dir = MODEL_DIR if os.path.isdir(MODEL_DIR) else "."
long_csv = os.path.join(out_dir, "loss_per_epoch_long.csv")
wide_csv = os.path.join(out_dir, "loss_per_epoch_wide_plus_test.csv")
best_csv = os.path.join(out_dir, "val_best_epoch_summary_plus_train.csv")

loss_long_df.to_csv(long_csv, index=False)
loss_wide_df.to_csv(wide_csv, index=False)
val_best_df.to_csv(best_csv, index=False)

# 7) Quick previews + LaTeX
print("\n=== Loss per Epoch (wide) + TestLoss ===")
print(loss_wide_df.to_string(index=False,
                             float_format=lambda x: f"{x:.6f}" if pd.notnull(x) else "NaN"))

print("\n=== Best Val Epoch Summary (incl. Train@Best, Test) ===")
print(val_best_df.to_string(index=False,
                            float_format=lambda x: f"{x:.6f}" if pd.notnull(x) else "NaN"))

print("\nLaTeX — Best Val Epoch Summary:")
print(val_best_df.to_latex(index=False, float_format="%.6f"))



